# Purpose: What cutting VNC do?

In [ ]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq
from mapd.sinq_builders import build_composite_sinq
import h5py
import glob
import numpy as np
import pickle
import plotly.express as px
import matplotlib.colors as mcolors
# import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)  # reset to defaults
mpl.rcParams['pdf.fonttype'] = 42         # embed fonts as text, not paths
mpl.rcParams['svg.fonttype'] = 'none'     # keep text editable in SVG
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 11

import seaborn as sns
from collections import defaultdict
import random

%load_ext autoreload
%autoreload 2

import mapd.sentinels  # load once
%aimport -mapd.sentinels

%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)

fig_dir = './FigureS2_vnc_cut'
os.makedirs(fig_dir, exist_ok=True)


# Load Figure 1 control sinq with VNC_cut flies

In [ ]:
plt.close('all')



In [ ]:
sinq = Sinq(sinqname='Fig1_control')
df_vnc_cut = sinq.display_df().loc[sinq.has_tag('vnc_cut')].copy()
df_vnc_cut['genotype'].unique().tolist()

In [ ]:
geno_map = {
 'ppk-Gal4;10XUAS-ChR in WT':       '+;ppk-GAL4;10XUAS-ChR',
 '+;ppk-GAL4;10XUAS-ChR':           '+;ppk-GAL4;10XUAS-ChR',
 'Hot-Cell-Gal4_ChrinWT':           '+;Hot-cell-GAL4;10XUAS-ChR',
}

df_vnc_cut['geno_smpl'] = df_vnc_cut['genotype'].map(geno_map)

In [ ]:
df_vnc_cut

# Average no_as trials before and after cuts, etc.

In [ ]:
def get_filtered_trial_average_probe(T,index = None,outcome_list = ['as_off','timeout','as_off_late','timeout_fail'],n_pre  = 100,n_post = 800):
    if index is None:
        index = T.df.index

    df = T.df.loc[index].copy()
    as_off_idx = df.loc[df['as_outcome'].isin(outcome_list)].index

    trials = df.loc[as_off_idx,'Trial']

    def dpp_snippet(tr):
        def closest_zero_idx(x):
            return int(np.nanargmin(np.abs(x)))

        dpp = tr.probe_position[tr.downsample_probe]-tr.probeZero
        x = tr.time[tr.downsample_probe]

        i0 = closest_zero_idx(x)
        x = x-x[i0]

        if i0 < n_pre or len(x) - i0 - 1 < n_post:
            raise ValueError("Not enough data around zero")
            
        x_win   = x[i0-n_pre : i0+n_post+1]
        dpp_win = dpp[i0-n_pre : i0+n_post+1]

        return pd.Series(
            {"x": x_win, "dpp": dpp_win}
        )
        # return x_win,dpp_win

    out = trials.apply(dpp_snippet)

    X = np.stack(out['x'].values)     # shape: (n_trials, 901)
    DPP = np.stack(out['dpp'].values) # shape: (n_trials, 901)

    x_mean   = np.nanmean(X, axis=0)
    dpp_mean = np.nanmean(DPP, axis=0)
    dpp_mean = np.squeeze(dpp_mean, axis=-1)
    dpp_std = np.nanstd(DPP, axis=0,ddof=1)

    n_trials = np.sum(~np.isnan(DPP), axis=0)
    dpp_sem = dpp_std / np.sqrt(n_trials)

    dpp_std = np.squeeze(dpp_std, axis=-1)
    dpp_sem = np.squeeze(dpp_sem, axis=-1)
    
    l = n_pre+n_post+1
    assert x_mean.shape == (l,)
    assert dpp_mean.shape == (l,)
    assert dpp_std.shape == (l,)
    assert dpp_sem.shape == (l,)


    df_summary = pd.DataFrame({
        "t": x_mean,        # shape (901,)
        "dpp_mean": dpp_mean,
        "dpp_std": dpp_std,
        'dpp_sem': dpp_sem
    })

    return df_summary


def plot_average_response(df_summary,dfc = None,fig_dir=None,path_name=None):
    fig = Figure(figsize=(4, 4), dpi=300)
    canvas = FigureCanvas(fig)
    ax = fig.add_subplot(111)

    t = df_summary['t']
    dpp_mean = df_summary['dpp_mean']
    dpp_std = df_summary['dpp_std']
    dpp_sem = df_summary['dpp_sem']

    ax.fill_between(
        t,
        dpp_mean - dpp_std,
        dpp_mean + dpp_std,
        color="lightgray",
        alpha=0.4,
        linewidth=0,
        label="± STD",
    )

    ax.fill_between(
        t,
        dpp_mean - dpp_sem,
        dpp_mean + dpp_sem,
        color="#8ecae6",   # light blue
        alpha=0.6,
        linewidth=0,
        label="± SEM",
    )

    ax.plot(
        t,
        dpp_mean,
        color="black",
        linewidth=2,
        label="Mean",
    )

    ax.axvline(0, color="k", linestyle="--")
    ax.set_title(f'Mean Response: {dfc}: {path_name}')
    ax.set_xlabel('time (s)')
    ax.set_ylabel('probe position (um)')
       
    if not fig_dir is None:
        export_path = f'{fig_dir}/exports'
        os.makedirs(export_path,exist_ok=True)
        canvas.print_png(f'{fig_dir}/average_response_{dfc}_{path_name}.png')
        df_summary.to_pickle(path=f'{export_path}/as_off_average_{dfc}_{path_name}.pkl')
        return fig


In [ ]:
for dfc in df_vnc_cut.index:
    sinq.restore_table(dayflycell=dfc)

## intact vs cut

In [ ]:
outdict = defaultdict(dict)

for dfc in df_vnc_cut.index: # '241029_F2_C1'
    T = sinq.restore_table(dayflycell=dfc)
    
    for cnd,ol in zip(['intact','cut'], [['as_off'],['as_off','timeout','as_off_late','timeout_fail']]):
        idx = T.df.index[T.df['vnc_status'] == cnd]
        if not idx.empty:
            out = get_filtered_trial_average_probe(T,index=idx,outcome_list=ol)
            fig = plot_average_response(df_summary=out,dfc=dfc,fig_dir=fig_dir,path_name=f'{dfc}_{cnd}')
            outdict[f'{dfc}'][f'{cnd}'] = out


# Red vs. Blue

In [ ]:
for dfc in df_vnc_cut.index: # '241029_F2_C1'
    T = sinq.restore_table(dayflycell=dfc)
    
    for stim in ['red','blue']:
        cnd = 0 if stim=='red' else 1
        idx = T.df.index[(T.df['vnc_status'] == 'intact') & (T.df['blueToggle']==cnd)]
        print(f'{dfc} has {len(idx)} {stim} trials')
        if not idx.empty:
            out = get_filtered_trial_average_probe(T,index=idx,outcome_list=['as_off'])
            fig = plot_average_response(df_summary=out,dfc=dfc,fig_dir=fig_dir,path_name=f'{dfc}_{stim}')
            outdict[f'{dfc}'][f'{stim}'] = out

In [ ]:
out

In [ ]:

def mean_across_dfcs(outdict, condition, value_col='dpp_mean'):
    """
    Average a given condition across all dfcs.
    Returns a DataFrame with averaged t and value_col.
    """
    dfs = []
    ts = []

    for dfc in outdict:
        if condition not in outdict[dfc]:
            continue

        df = outdict[dfc][condition]

        ts.append(df[['t']].rename(columns={'t': dfc}))
        dfs.append(df[[value_col]].rename(columns={value_col: dfc}))

    # align on index and average row-wise
    t_stack = pd.concat(ts, axis=1)
    val_stack = pd.concat(dfs, axis=1)

    mean_df = pd.DataFrame(
        index=val_stack.index,
        data={
            't': t_stack.mean(axis=1),
            value_col: val_stack.mean(axis=1),
            f'{value_col}_std': val_stack.std(axis=1),
        }
    )

    return mean_df

intact_mean = mean_across_dfcs(outdict, 'intact')
cut_mean    = mean_across_dfcs(outdict, 'cut')

In [ ]:
outdict['MEAN'] = {
    'intact': intact_mean,
    'cut': cut_mean
}

In [ ]:
fig = Figure(figsize=(8,4), dpi=300)
canvas = FigureCanvas(fig)

conditions = ['intact', 'cut', 'red', 'blue']

colors = [
    "#1f77b4",  # blue
    "#ff7f0e",  # orange
    "#2ca02c",  # green
    "#d62728",  # red
    "#9467bd",  # purple
    "#8c564b",  # brown         
    "#e377c2",  # pink
    "#000000"
]

color_map = dict(zip(outdict.keys(), colors))

axes = []
for i, cond in enumerate(conditions):
    ax = fig.add_subplot(1, len(conditions), i + 1)
    axes.append(ax)

    for dfc in outdict.keys():
        if not cond in outdict[dfc].keys():
            continue
        out = outdict[dfc][cond]
        ax.plot(out['t'], out['dpp_mean'], label=dfc,color=color_map[dfc])

    ax.set_title(cond)
    ax.set_ylabel("dpp_mean")
    ax.set_ylim([-500,10])

axes[-1].set_xlabel("x")

# single legend (optional)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right")

fig.tight_layout()
display(fig)

fig.savefig(f'{fig_dir}/as_off_resp_{'_'.join(conditions)}')

fig.savefig(f'{fig_dir}/as_off_resp_{'_'.join(conditions)}.svg',format='svg')

# See if they are still moving after the VNC is cut

In [ ]:
T.extract_trial_properties()
T.compute_trial_method('probe_positive_effort')
fig, ax = T.plot_trial_computations(method_name='probe_positive_effort')
display(fig)
fig.savefig(f'{fig_dir}/positive_probe_effort_241105_F3_C1.svg')

In [ ]:
fig = Figure(figsize=(6, 4), dpi=300)
canvas = FigureCanvas(fig)
ax = fig.add_subplot(111)
fig,ax = T.plot_some_probe_groups(index=T.df.loc[135:145].index,fig = fig,ax=ax)

ax.set_ylim([-600, 20])
display(fig)
fig.savefig(f'{fig_dir}/cut_movement_example_{T.dfc}_135_145.svg')

fig = Figure(figsize=(6, 4), dpi=300)
canvas = FigureCanvas(fig)
ax = fig.add_subplot(111)
fig,ax = T.plot_some_probe_groups(index=T.df.loc[137:142].index,fig = fig,ax=ax)

ax.set_ylim([-600, 20])
display(fig)
fig.savefig(f'{fig_dir}/cut_movement_example_{T.dfc}_137_142.svg')

In [ ]:
fig = Figure(figsize=(6, 4), dpi=300)
canvas = FigureCanvas(fig)
ax = fig.add_subplot(111)
fig,ax = T.plot_some_probe_groups(index=T.df.loc[234:238].index,fig = fig,ax=ax)

display(fig)
fig.savefig(f'{fig_dir}/cut_movement_example_{T.dfc}_234_238.svg')

fig = Figure(figsize=(6, 4), dpi=300)
canvas = FigureCanvas(fig)
ax = fig.add_subplot(111)
fig,ax = T.plot_some_probe_groups(index=T.df.loc[235:237].index,fig = fig,ax=ax)

display(fig)
fig.savefig(f'{fig_dir}/cut_movement_example_{T.dfc}_235_237.svg')

In [ ]:
T.df.loc[230:240].index